## 卷积神经网络：验证码识别

欢迎来到验证码识别项目！在此文件中，有些示例代码已经提供给你，但你还需要实现更多的功能来让项目成功运行。除非有明确要求，你无须修改任何已给出的代码。以 <img src='http://imgbed.momodel.cn/5cc1a0b8e3067ce9b6abf76f.jpg' width=16px height=16px> **编程练习**开始的标题表示接下来的内容中有需要你实现的功能。需要实现的部分也会在注释中以**TODO**标出。请仔细阅读所有的提示！你可以点击**问题提示**，查看每一部分详细的提示指导，也可以点击**插入答案**，把正确答案插入到下方代码块中。

除了实现代码外，你还需要回答一些与项目和实现有关的问题。每一个需要你回答的问题都会以 <img src='http://imgbed.momodel.cn/5cc1a0b8e3067ce9b6abf76e.jpg' width=16px height=16px>**思考问答**为标题。请仔细阅读每个问题，作出答复。当然我们也为你提供**问题提示**和**查看答案**的按钮。

你可以通过单击代码区域，然后使用键盘快捷键 **Shift+Enter** 或 **Shift+ Return** 来运行代码。或者在选择代码后使用**播放**（run cell）按钮执行代码。像这样的 MarkDown 文本可以通过双击编辑，并使用这些相同的快捷键保存。

**文档中提供的代码具有顺序性，必须从前往后依次运行代码，不能跳跃执行，否则可能出现意想不到的错误！**

---
### 第一步：项目概述


目前随着深度学习，越来越蓬勃的发展，在图像识别和语音识别中也表现出了强大的生产力。
但是对于我们来说，经常去跑那些公开的大型数据库，比如ImageNet或者CoCo，可以会觉得学到的这个屠龙之技离我们自己的生活好遥远。
那么我们本次实战，就是希望将此技术运用到一些大家在日常生活中就能感知的场景上。


大家在很多网站上都会遇到“验证码”，比如下图中的这种。验证码每次都要让人去填写，的确很麻烦，那么我们能不能用深度学习的方法来自动验证这个验证码呢？

![](http://imgbed.momodel.cn/5cc1a0b8e3067ce9b6abf771.jpg)

captcha 是 python 的一个生成验证码图片的工具包，我们使用它来生成验证码图片。

In [ ]:
import subprocess
import sys

# 安装必要包
def install_dependencies():
    required_packages = ["captcha", "tensorflow", "matplotlib", "numpy", "pillow"]
    for pkg in required_packages:
        try:
            __import__(pkg)
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

install_dependencies()


In [ ]:
# 2. 导入核心库
import tensorflow as tf
import numpy as np
from captcha.image import ImageCaptcha
import matplotlib.pyplot as plt
import random
import string
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Lambda, Reshape, Dense,
                                     BatchNormalization, Activation, Dropout,
                                     GRU, add, Conv2D, MaxPooling2D)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import Callback
from collections import Counter

# 配置matplotlib显示
%matplotlib inline
%config InlineBackend.figure_format = 'retina'


<img src='http://imgbed.momodel.cn/5cc1a0b8e3067ce9b6abf76f.jpg' width=16px height=16px>  **编程练习**

我们的标签数目应该是多少？请补充下面的代码

In [ ]:
# TODO 0
n_class = None


<span class='md-answer-link pop 0'>问题提示</span> <span class='md-answer-link insert 0'>插入答案</span>

### 第二步：生成验证码图片


In [ ]:
# 验证码字符集（数字0-9 + 大写字母A-Z）
characters = string.digits + string.ascii_uppercase
# 类别数（字符集长度 + 1，1为CTC算法的空白字符）
n_class = len(characters) + 1
# 验证码图像尺寸（宽、高）和字符长度
width, height, n_len = 170, 80, 4
# GRU网络单元数量
rnn_size = 128

# 我们打印一张生成出来的验证码的图片看看
generator = ImageCaptcha(width=width, height=height)
random_str = ''.join([random.choice(characters) for j in range(4)])
img = generator.generate_image(random_str)


plt.imshow(img)
plt.title(random_str)
plt.show()


### 第三步：了解 CTC 损失函数

#### CTC 损失函数

这个 loss 是一个特别神奇的 loss，它可以在只知道序列的顺序，不知道具体位置的情况下，让模型收敛。（具体可以参考[warp-ctc](https://github.com/baidu-research/warp-ctc)）

![](http://imgbed.momodel.cn/5cc1a0b8e3067ce9b6abf770.jpg)

在 Keras 里面，CTC Loss 已经内置了，我们直接定义这样一个函数即可，由于我们使用的是循环神经网络，所以默认丢掉前面两个输出，因为它们通常无意义，且会影响模型的输出。

y_pred 是模型的输出，是按顺序输出的37个字符的概率，因为我们这里用到了循环神经网络，所以需要一个空白字符的类；

labels 是验证码，是四个数字，每个数字对应字符的编号；

input_length 表示 y_pred 的长度，我们这里是15；

label_length 表示 labels 的长度，我们这里是4。

### 第四步：模型实现


我们的模型结构是这样设计的，

![](./model.png)

首先我们通过多层卷积神经网络去识别验证码图片中的特征，大家可以看到，从 Input 到 最后一个 MaxPooling2D，是一个很深的卷积神经网络，它负责学习字符的各个特征，尽可能区分不同的字符。它输出 shape 是 [None, 17, 6, 128]，这个形状相当于把一张宽为 170，高为 80 的彩色图像 (170, 80, 3)，压缩为宽为 17，高为 6 的 128维特征的特征图 (17, 6, 128)。

然后我们把图像 reshape 成 (17, 768)，也就是把高和特征放在一个维度，然后降维成 (17, 128)，也就是从左到右有17条特征，每个特征128个维度。这128个维度可以说就是这一个图像的非常高维，非常抽象的概括。

然后我们将17个特征向量依次输入到 GRU（特殊的循环神经网络）中，你可以把GRU理解为是 LSTM 的简化版。LSTM 早在1997年就已经被发明出来了，但是 GRU 直到2014年才出现。经过实验，GRU 效果比 LSTM 要好。所以我们这里使用 GRU。

最后 Dropout 接一个全连接层，作为分类器输出每个字符的概率。

这个就是我们 base_model 的结构，也就是我们模型的结构。后面的 labels, input_length, label_length 和 loss_out 都是为了输入必要的数据来计算 CTC Loss 的。

<img src='http://imgbed.momodel.cn/5cc1a0b8e3067ce9b6abf76f.jpg' width=16px height=16px>  **编程练习**

根据上面的模型图，建立卷积层

<span class='md-hint-alone-link insert 1'>插入答案</span>

In [ ]:
# 数据生成器（核心修复：适配TensorFlow 2.x张量结构）
def gen(batch_size=128):
    # 初始化验证码生成器
    captcha_generator = ImageCaptcha(width=width, height=height)

    # 定义输出签名：明确每个张量的形状和类型（严格匹配返回值）
    output_signature = (
        # 输入部分：(图像张量, 标签张量, 输入长度张量, 标签长度张量)
        (
            tf.TensorSpec(shape=(batch_size, width, height, 3), dtype=tf.uint8),
            tf.TensorSpec(shape=(batch_size, n_len), dtype=tf.uint8),
            tf.TensorSpec(shape=(batch_size,), dtype=tf.uint8),
            tf.TensorSpec(shape=(batch_size,), dtype=tf.uint8)
        ),
        # 输出部分：CTC损失计算的占位符（值无意义，仅匹配格式）
        tf.TensorSpec(shape=(batch_size,), dtype=tf.float32)
    )

    def generator_fn():
        while True:  # 无限生成（满足训练批次循环需求）
            # 初始化批次数据数组
            X = np.zeros((batch_size, width, height, 3), dtype=np.uint8)  # 图像数据
            y = np.zeros((batch_size, n_len), dtype=np.uint8)              # 标签（字符索引）

            # 生成单批次内的每个样本
            for i in range(batch_size):
                # 生成随机4位验证码字符串
                random_str = ''.join([random.choice(characters) for _ in range(n_len)])
                # 生成验证码图像并转换为数组（调整维度为：宽×高×3）
                img = captcha_generator.generate_image(random_str)
                X[i] = np.array(img).transpose(1, 0, 2)  # 适配模型输入维度
                # 将字符转换为索引（如'A'→10，'0'→0）
                y[i] = [characters.find(c) for c in random_str]

            # 转换为TensorFlow张量（关键：解决类型不匹配问题）
            X_tensor = tf.convert_to_tensor(X, dtype=tf.uint8)
            y_tensor = tf.convert_to_tensor(y, dtype=tf.uint8)
            # 输入长度（RNN序列长度，提前计算后固定）
            input_len_tensor = tf.convert_to_tensor(
                np.ones(batch_size, dtype=np.uint8) * rnn_length,
                dtype=tf.uint8
            )
            # 标签长度（固定为4，即验证码字符数）
            label_len_tensor = tf.convert_to_tensor(
                np.ones(batch_size, dtype=np.uint8) * n_len,
                dtype=tf.uint8
            )
            # 占位符输出（CTC损失计算用，值为1不影响结果）
            dummy_output = tf.convert_to_tensor(
                np.ones(batch_size, dtype=np.float32),
                dtype=tf.float32
            )

            # 返回元组结构（严格匹配output_signature，避免列表导致的错误）
            yield (
                (X_tensor, y_tensor, input_len_tensor, label_len_tensor),
                dummy_output
            )

    # 包装为TensorFlow数据集（带签名）
    return tf.data.Dataset.from_generator(
        generator_fn,
        output_signature=output_signature
    )




<img src='http://imgbed.momodel.cn/5cc1a0b8e3067ce9b6abf76f.jpg' width=16px height=16px>  **编程练习**

仿照上面的 GRU 的构建，建立第二层的 GRU

In [ ]:
# TODO 2
gru_2 = None
gru_2b = None
x = None


<span class='md-hint-alone-link insert 2'>插入答案</span>

In [ ]:
# 构建CNN+双向GRU模型
# 输入层（图像：宽×高×3通道）
input_tensor = Input(shape=(width, height, 3), name="image_input")
x = input_tensor

# 数据归一化（将像素值从[0,255]转为[-1,1]，加速训练）
x = Lambda(
    lambda x: (tf.cast(x, tf.float32) - 127.5) / 127.5,
    name="normalization"
)(x)

# 卷积层（特征提取：3层CNN+池化，逐步缩小尺寸、增加通道数）
# 第一层卷积
x = Conv2D(32, (3, 3), activation="relu", padding="same", name="conv1")(x)
x = MaxPooling2D(pool_size=(2, 2), name="pool1")(x)  # 尺寸减半
# 第二层卷积
x = Conv2D(64, (3, 3), activation="relu", padding="same", name="conv2")(x)
x = MaxPooling2D(pool_size=(2, 2), name="pool2")(x)  # 尺寸再减半
# 第三层卷积
x = Conv2D(128, (3, 3), activation="relu", padding="same", name="conv3")(x)
x = MaxPooling2D(pool_size=(2, 2), name="pool3")(x)  # 尺寸再减半

# 计算RNN输入维度（修复shape获取：元组转列表，兼容所有TF版本）
conv_shape = list(x.shape)  # 格式：[None, 卷积输出宽, 卷积输出高, 通道数]
rnn_length = conv_shape[1]  # RNN序列长度（卷积输出的宽度）
rnn_dimen = conv_shape[2] * conv_shape[3]  # RNN输入维度（高×通道数）
print(f"卷积层输出形状: {conv_shape}")
print(f"RNN序列长度: {rnn_length}, RNN输入维度: {rnn_dimen}")

# 适配RNN输入格式（将2D特征图转为1D序列）
x = Reshape(
    target_shape=(rnn_length, rnn_dimen),
    name="reshape_for_rnn"
)(x)
rnn_length -= 2  # 调整RNN长度（匹配CTC解码时的序列截断需求）

# 全连接层（将RNN输入维度统一为rnn_size）
x = Dense(rnn_size, kernel_initializer="he_uniform", name="dense_before_rnn")(x)
x = BatchNormalization(name="bn_before_rnn")(x)  # 批量归一化，加速训练
x = Activation("relu", name="relu_before_rnn")(x)
x = Dropout(0.2, name="dropout_before_rnn")(x)  # 防止过拟合

# 双向GRU层（特征序列建模，2层双向结构提升性能）
# 第一层双向GRU
gru1 = GRU(rnn_size, return_sequences=True, kernel_initializer="he_uniform", name="gru1")(x)
gru1_back = GRU(rnn_size, return_sequences=True, kernel_initializer="he_uniform",
                go_backwards=True, name="gru1_back")(x)
x = add([gru1, gru1_back], name="add_gru1")  # 拼接正向和反向GRU输出

# 第二层双向GRU
gru2 = GRU(rnn_size, return_sequences=True, kernel_initializer="he_uniform", name="gru2")(x)
gru2_back = GRU(rnn_size, return_sequences=True, kernel_initializer="he_uniform",
                go_backwards=True, name="gru2_back")(x)
x = add([gru2, gru2_back], name="add_gru2")  # 拼接正向和反向GRU输出

# 输出层（CTC分类：每个时间步输出n_class个类别概率）
x = Dropout(0.2, name="dropout_before_output")(x)
output_tensor = Dense(n_class, activation="softmax", name="output_layer")(x)

# 基础模型（仅用于预测，输入图像→输出类别概率）
base_model = Model(inputs=input_tensor, outputs=output_tensor, name="base_predict_model")


### 第五步：构造图片生成器


根据模型的输入，我们需要输入四个数据：

X 是一批图片；

y 是每个图片对应的 label，最大长度为 n_len；

input_length 表示模型输出的长度，我们这里是15；

label_length 表示 labels 的长度，我们这里是4。

最后还有一个输入是 np.ones(batch_size)，这是因为 Keras 在训练模型的时候必须输入一个 X 和一个 y，我们这里把上面四个都合并为一个 X 了，因此实际上 y 没有参与 loss 的计算，所以随便使用一个 batch_size。

In [ ]:
# 构建带CTC损失的训练模型
# CTC损失需要的额外输入（标签、输入长度、标签长度）
labels_input = Input(shape=[n_len], dtype=tf.float32, name="the_labels")
input_length_input = Input(shape=[1], dtype=tf.int64, name="input_length")
label_length_input = Input(shape=[1], dtype=tf.int64, name="label_length")

# 定义CTC损失函数（Lambda层封装，适配Keras接口）
def ctc_lambda_func(args):
    y_pred, labels, input_length, label_length = args
    y_pred = y_pred[:, 2:, :]  # 截断前2个时间步（去除无效特征）
    return K.ctc_batch_cost(labels, y_pred, input_length, label_length)

# CTC损失层
ctc_loss = Lambda(
    ctc_lambda_func,
    output_shape=(1,),
    name="ctc_loss_layer"
)([output_tensor, labels_input, input_length_input, label_length_input])

# 训练模型（输入：图像+标签+长度信息；输出：CTC损失）
train_model = Model(
    inputs=[input_tensor, labels_input, input_length_input, label_length_input],
    outputs=ctc_loss,
    name="train_model_with_ctc"
)

# 编译模型（CTC损失特殊处理：y_true不参与计算，直接返回y_pred）
train_model.compile(
    loss={"ctc_loss_layer": lambda y_true, y_pred: y_pred},
    optimizer=Adam(learning_rate=1e-3)
)



### 第六步：评估模型

我们会通过下面这个函数来评估我们的模型，和上面的评估标准一样，只有全部正确，我们才算预测正确。

这里需要注意的是，模型最开始训练的时候，并不一定会输出四个字符，所以我们如果遇到所有的字符都不到四个的时候，就不用计算了，一定是全错。遇到多于四个字符的时候，只取前四个。

In [ ]:
# 定义评估函数（计算模型准确率）
def evaluate_model(batch_size=128, steps=10):
    total_acc = 0.0
    data_generator = gen(batch_size)

    for _ in range(steps):
        # 获取单批次测试数据（解包元组结构）
        (X_test, y_test, _, _), _ = next(iter(data_generator))
        # 模型预测（输出每个时间步的类别概率）
        y_pred = base_model.predict(X_test, verbose=0)
        # CTC解码（从概率序列中提取最可能的字符序列）
        decode_shape = y_pred[:, 2:, :].shape  # 截断前2个时间步
        ctc_decode = K.ctc_decode(
            y_pred[:, 2:, :],
            input_length=np.ones(decode_shape[0]) * decode_shape[1]
        )[0][0]  # 获取解码结果（去除空白字符）
        # 转换为NumPy数组（方便计算准确率）
        pred_indices = K.get_value(ctc_decode)[:, :n_len]  # 取前4个字符（验证码长度）
        # 计算当前批次准确率（所有字符都正确才算一个正确样本）
        y_test_np = y_test.numpy()  # Tensor转NumPy数组
        batch_acc = (y_test_np == pred_indices).all(axis=1).mean()
        total_acc += batch_acc

    # 返回平均准确率
    return total_acc / steps


### 第七步： 评估回调函数的定义


因为 Keras 没有针对 CTC 模型计算准确率的选项，因此我们需要自定义一个回调函数，它会在每一代训练完成的时候计算模型的准确率。

In [ ]:
# 定义训练回调（每轮结束后评估准确率）
class AccuracyEvaluator(Callback):
    def __init__(self):
        super().__init__()
        self.acc_history = []  # 保存每轮准确率

    def on_epoch_end(self, epoch, logs=None):
        # 计算当前轮准确率
        current_acc = evaluate_model(steps=20) * 100
        self.acc_history.append(current_acc)
        # 打印准确率（保留2位小数）
        print(f"\n【Epoch {epoch+1}】 验证准确率: {current_acc:.2f}%\n")

# 初始化评估器
evaluator = AccuracyEvaluator()


### 第八步：训练模型

我们先按 `Adam(1e-3)` 的学习率训练20代，让模型快速收敛，然后以 `Adam(1e-4)` 的学习率再训练20代。这里设置每代训练 400 个 step，也就是每代 400*128=51200 个样本，验证集设置的是 20*128=2048 个样本。

In [ ]:
# 开始训练（分两轮：先高学习率快速收敛，再低学习率精细优化）
print("="*50)
print("开始第一轮训练（学习率1e-3，共20轮）")
print("="*50)
# 第一轮训练（学习率1e-3）
history1 = train_model.fit(
    gen(batch_size=128),          # 训练数据生成器
    steps_per_epoch=400,          # 每轮训练批次数量
    epochs=20,                    # 训练轮数
    callbacks=[evaluator],        # 准确率评估回调
    validation_data=gen(batch_size=128),  # 验证数据生成器
    validation_steps=20,          # 每轮验证批次数量
    verbose=1                     # 显示训练进度（1：显示进度条，0：不显示）
)

# 调整学习率为1e-4，继续训练（精细优化）
print("="*50)
print("开始第二轮训练（学习率1e-4，共20轮）")
print("="*50)
train_model.compile(
    loss={"ctc_loss_layer": lambda y_true, y_pred: y_pred},
    optimizer=Adam(learning_rate=1e-4)
)
history2 = train_model.fit(
    gen(batch_size=128),
    steps_per_epoch=400,
    epochs=20,
    callbacks=[evaluator],
    validation_data=gen(batch_size=128),
    validation_steps=20,
    verbose=1
)


<img src='http://imgbed.momodel.cn/5cc1a0b8e3067ce9b6abf76e.jpg' width=16px height=16px> **思考问答**

查看 20 代 和 40 代时模型的表现。acc 分别达到了多少？

<span class='md-hint-alone-link pop 3'>查看答案</span>

### 第九步：测试模型
接下来，我们对模型进行一些测试。

In [ ]:
print("="*50)
print("可视化预测结果（12个样本）")
print("="*50)
# 获取12个测试样本
test_data = next(iter(gen(batch_size=12)))
(X_vis, y_vis, _, _), _ = test_data

# 模型预测与解码
y_pred_vis = base_model.predict(X_vis, verbose=0)
decode_shape_vis = y_pred_vis[:, 2:, :].shape
ctc_decode_vis = K.ctc_decode(
    y_pred_vis[:, 2:, :],
    input_length=np.ones(decode_shape_vis[0]) * decode_shape_vis[1]
)[0][0]
pred_indices_vis = K.get_value(ctc_decode_vis)[:, :n_len]

# 绘制结果（3行4列布局）
plt.figure(figsize=(16, 10))
for i in range(12):
    plt.subplot(3, 4, i+1)
    # 转换图像维度为（高×宽×3）以正常显示
    img_show = tf.transpose(X_vis[i], (1, 0, 2)).numpy()
    plt.imshow(img_show)
    # 转换索引为字符（处理无效索引）
    pred_str = ''.join([
        characters[x] if 0 <= x < len(characters) else '?'
        for x in pred_indices_vis[i]
    ])
    real_str = ''.join([characters[x] for x in y_vis[i].numpy()])
    # 设置标题（绿色：正确，红色：错误）
    color = "green" if pred_str == real_str else "red"
    plt.title(f"预测: {pred_str}\n真实: {real_str}", color=color, fontsize=12)
    plt.axis("off")  # 隐藏坐标轴
plt.tight_layout()
plt.show()


### 第十步：评估模型


我们可以尝试计算模型的总体准确率，以及看看模型到底错在哪。
首先生成1024个样本，然后用 base_model 进行预测，然后裁剪并进行 ctc 解码，最后裁剪到4个 label 并与真实值进行对比。

In [ ]:
(X_vis, y_vis, input_length_vis, label_length_vis), _ = next(gen(10000))

y_pred = base_model.predict(X_vis, verbose=1)
shape = y_pred[:,2:,:].shape
ctc_decode = K.ctc_decode(y_pred[:,2:,:], input_length=np.ones(shape[0])*shape[1])[0][0]
out = K.get_value(ctc_decode)[:, :4]
# 大规模测试（10000样本）与错误统计
print("="*50)
print("大规模测试（10000样本）与错误分析")
print("="*50)

# 注意：若显存不足，可将batch_size从10000改为128/256，分多次预测后合并（此处用10000需足够显存）
batch_size_large = 10000  # 单次生成10000个样本（根据显存调整，建议≤2048）

# 获取10000个测试样本（使用之前定义的gen生成器）
try:
    large_test_data = next(iter(gen(batch_size=batch_size_large)))
    (X_test_large, y_test_large, _, _), _ = large_test_data
    print(f"成功加载 {batch_size_large} 个测试样本")
except Exception as e:
    # 若单次生成10000样本显存不足，自动降级为分批次生成（每批128）
    print(f"单次生成{batch_size_large}样本失败，自动降级为分批次处理：{str(e)}")
    batch_size_large = 128
    total_samples = 10000
    # 初始化空数组存储所有样本
    X_test_large = np.zeros((total_samples, width, height, 3), dtype=np.uint8)
    y_test_large = np.zeros((total_samples, n_len), dtype=np.uint8)

    # 分批次生成并拼接
    for i in range(total_samples // batch_size_large):
        batch_data = next(iter(gen(batch_size=batch_size_large)))
        (X_batch, y_batch, _, _), _ = batch_data
        start_idx = i * batch_size_large
        end_idx = start_idx + batch_size_large
        X_test_large[start_idx:end_idx] = X_batch.numpy()
        y_test_large[start_idx:end_idx] = y_batch.numpy()
    print(f"分 {total_samples//batch_size_large} 批加载完成 {total_samples} 个测试样本")

# 模型预测（verbose=1显示进度条）
print("\n开始模型预测...")
if isinstance(X_test_large, tf.Tensor):
    X_test_large = X_test_large.numpy()  # 确保输入为NumPy数组（兼容部分TF版本）
y_pred_large = base_model.predict(X_test_large, verbose=1)

# CTC解码：从预测概率中提取字符索引
print("\n开始CTC解码...")
decode_shape_large = y_pred_large[:, 2:, :].shape  # 截断前2个无效时间步
# 解码（CTC自动去除空白字符）
ctc_decode_large = K.ctc_decode(
    y_pred_large[:, 2:, :],  # 输入预测概率（截断后）
    input_length=np.ones(decode_shape_large[0]) * decode_shape_large[1]  # 每个样本的输入长度
)[0][0]  # 获取解码后的索引序列

# 转换解码结果为NumPy数组，并截取前n_len个字符（确保与验证码长度一致）
pred_indices_large = K.get_value(ctc_decode_large)[:, :n_len]

# 转换真实标签为NumPy数组（若为Tensor）
if isinstance(y_test_large, tf.Tensor):
    y_test_large = y_test_large.numpy()

# 11.1 计算总体准确率
# 规则：每个样本的4个字符全部预测正确，才算“正确样本”
correct_samples = (y_test_large == pred_indices_large).all(axis=1).sum()
total_samples = len(y_test_large)
overall_accuracy = correct_samples / total_samples

print(f"\n【大规模测试结果】")
print(f"总样本数：{total_samples}")
print(f"正确样本数：{correct_samples}")
print(f"总体准确率：{overall_accuracy:.4f} ({correct_samples}/{total_samples})")





查看模型在哪些字符上错误次数最多？

In [ ]:
# 统计错误字符对（真实字符 → 预测字符）
print("\n【错误字符分析】")
error_char_pairs = []  # 存储所有错误的“真实→预测”字符对
error_sample_count = 0  # 错误样本总数

# 遍历每个样本，分析错误
for idx in range(total_samples):
    true_indices = y_test_large[idx]
    pred_indices = pred_indices_large[idx]

    # 判断当前样本是否错误
    if not (true_indices == pred_indices).all():
        error_sample_count += 1
        # 遍历每个字符位置，记录错误对
        for true_idx, pred_idx in zip(true_indices, pred_indices):
            if true_idx != pred_idx:
                # 转换索引为字符（处理无效索引，避免越界）
                true_char = characters[true_idx] if 0 <= true_idx < len(characters) else "?"
                pred_char = characters[pred_idx] if 0 <= pred_idx < len(characters) else "?"
                error_char_pairs.append(f"{true_char}→{pred_char}")

# 计算错误样本率
error_sample_rate = error_sample_count / total_samples
print(f"错误样本数：{error_sample_count}")
print(f"错误样本率：{error_sample_rate:.4f}")

# 11.3 输出Top10高频错误字符对
if error_char_pairs:
    # 使用Counter统计频率
    error_counter = Counter(error_char_pairs)
    print(f"\nTop 10 高频错误字符对（真实→预测）：")
    for rank, (pair, count) in enumerate(error_counter.most_common(10), 1):
        # 计算该错误对占总错误的比例
        error_ratio = count / len(error_char_pairs)
        print(f"第{rank:2d}名：{pair:6s} → 出现{count:3d}次（占错误总数的{error_ratio:.2%}）")
else:
    print("\n无错误样本，所有字符预测正确！")

# 11.4 可视化错误样本（随机展示5个错误样本，若存在）
if error_sample_count > 0:
    print(f"\n【错误样本可视化】（随机展示5个）")
    # 获取所有错误样本的索引
    error_sample_indices = [
        idx for idx in range(total_samples)
        if not (y_test_large[idx] == pred_indices_large[idx]).all()
    ]
    # 随机选择5个错误样本（若不足5个则展示全部）
    selected_error_indices = random.sample(error_sample_indices, min(5, error_sample_count))

    # 绘制错误样本
    plt.figure(figsize=(15, 3))
    for plot_idx, sample_idx in enumerate(selected_error_indices, 1):
        plt.subplot(1, 5, plot_idx)
        # 读取错误样本图像并调整维度（适配显示）
        img_error = X_test_large[sample_idx]
        img_error_show = img_error.transpose(1, 0, 2)  # 从（宽×高×3）转为（高×宽×3）
        plt.imshow(img_error_show)
        # 转换索引为字符
        true_str = ''.join([characters[idx] for idx in y_test_large[sample_idx]])
        pred_str = ''.join([
            characters[idx] if 0 <= idx < len(characters) else "?"
            for idx in pred_indices_large[sample_idx]
        ])
        # 标题标注真实值和预测值（红色突出错误）
        plt.title(f"真实: {true_str}\n预测: {pred_str}", color="red", fontsize=10)
        plt.axis("off")  # 隐藏坐标轴
    plt.tight_layout()
    plt.show()
else:
    print("\n无错误样本可可视化")
(y_vis == out).all(axis=1).mean()
